# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the CPTAC files in `data/part1_cptac_brca/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [17]:
import sys
print(sys.executable)

/opt/anaconda3/envs/biot6900/bin/python


In [18]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
import numpy as np
import pandas as pd
from scipy import stats

In [20]:


RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [21]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [22]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/part1_cptac_brca/cptac_brca_rna.tsv", "prot": "data/part1_cptac_brca/cptac_brca_protein.tsv",
         "mut": "data/part1_cptac_brca/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/part1_cptac_brca/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [23]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/part1_cptac_brca/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [24]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [25]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [26]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [27]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save the raw downloads in `data/part2_cptac_brca_raw/`, then save the processed matrices into `data/part1_cptac_brca/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/part1_cptac_brca/cptac_brca_rna.tsv`
   - `data/part1_cptac_brca/cptac_brca_protein.tsv`
   - `data/part1_cptac_brca/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 —  Type 2 diabetes 

The three tables in `data/part3_t2d/` are **gene-level summaries from three different cohorts** — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

| Layer | File | Source | Effect column |
|---|---|---|---|
| Transcriptomic | `data/part3_t2d/t2d_rna.tsv` | Pancreatic islets from 103 organ donors, 19 T2D vs 84 non-diabetic (GEO **GSE76894**, Solimena et al. 2018) | `log2fc` (T2D − non-diabetic) |
| Proteomic | `data/part3_t2d/t2d_protein.tsv` | ~1,460 plasma proteins vs incident T2D in 47,600 UK Biobank participants (Gadd et al., *Nature Aging* 2024) | `log2hr` (log2 hazard ratio per SD of protein) |
| Genomic | `data/part3_t2d/t2d_gwas.tsv` | Open Targets GWAS credible-set evidence for T2D (`MONDO_0005148`) | `gwas_score` (0–1) |

The tables were built by `prepare_t2d_data.py` (run it once from the repo root to re-download and rebuild them).

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Expect* known T2D genes such as **LPL**, **PPARG**, **TCF7L2**, **SLC30A8** or **KCNJ11** to show up in at least one layer.

### 3.1 — TODO: load and inspect the three tables
Load the three T2D tables and look at each table's columns and shape before you touch them.

In [28]:
# TODO 3.1 — load the three T2D tables and inspect them.
tx = pd.read_csv("data/part3_t2d/t2d_rna.tsv", sep="\t")
pr = pd.read_csv("data/part3_t2d/t2d_protein.tsv", sep="\t")
gw = pd.read_csv("data/part3_t2d/t2d_gwas.tsv", sep="\t")

print("transcriptomics", tx.shape, list(tx.columns))
print("proteomics     ", pr.shape, list(pr.columns))
print("genomics       ", gw.shape, list(gw.columns))

transcriptomics (14077, 3) ['gene', 'log2fc', 'pval']
proteomics      (1458, 3) ['gene', 'log2hr', 'pval']
genomics        (3319, 2) ['gene', 'gwas_score']


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge on `gene`. RNA and protein are joined with an **inner** join (a gene needs both measurements). The GWAS table only lists genes *with* GWAS evidence, so it is joined with a **left** join and missing genes get `gwas_score = 0` (no evidence) instead of being dropped. Report how many genes survive the join (your first reality check).

In [29]:
tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2hr": "prot_lhr", "pval": "prot_p"})
df = tx.merge(pr, on="gene", how="inner").merge(gw, on="gene", how="left")
df["gwas_score"] = df["gwas_score"].fillna(0.0)  # not in the GWAS table = no GWAS evidence
print(f"Genes surviving the RNA + protein join: {len(df)} "
      f"(RNA {len(tx)}, protein {len(pr)}, GWAS {len(gw)})")
print(f"  of which have GWAS evidence: {(df['gwas_score'] > 0).sum()}")

Genes surviving the RNA + protein join: 1024 (RNA 14077, protein 1458, GWAS 3319)
  of which have GWAS evidence: 180


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Here that means *higher in T2D islets* and *higher plasma level → higher T2D risk* (or both lower). Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [30]:
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lhr"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)}")

Sign-concordant genes: 557 / 1024


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [31]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"]      = df["prot_lhr"].abs()
df["genomic"]        = df["gwas_score"]
df["score"] = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
print(f"Scored {len(df)} genes; score range [{df['score'].min():.3f}, {df['score'].max():.3f}]")

Scored 1024 genes; score range [0.155, 0.927]


### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_t2d.csv`** — this file is the hand-off to Week 3. Check where known T2D genes (LPL, PPARG, TCF7L2, SLC30A8, KCNJ11 …) land, and look at any `concordant == False` genes in your top hits.

In [32]:
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
top = ranked.head(15)
ranked.to_csv("targets_t2d.csv", index=False)
print(top[["gene", "rna_lfc", "prot_lhr", "gwas_score", "concordant", "score"]].round(3).to_string(index=False))

print("\nRank of known T2D genes (out of", len(ranked), "):")
for g in ["LPL", "PPARG", "TCF7L2", "SLC30A8", "KCNJ11", "IGFBP1", "LEP"]:
    hit = ranked.index[ranked["gene"] == g]
    print(f"  {g:<8}", hit[0] + 1 if len(hit) else "not in joined table")

     gene  rna_lfc  prot_lhr  gwas_score  concordant  score
      LPL    0.328    -0.837       0.921       False  0.927
   BAIAP2   -0.413     0.832       0.353       False  0.925
      CRH   -0.877    -0.515       0.299        True  0.912
     CSF1    0.339     0.660       0.595        True  0.900
   IL17RB   -0.495    -0.578       0.174        True  0.899
   IFNLR1    0.393     0.516       0.620        True  0.895
    ROBO1   -0.332     0.678       0.418       False  0.891
   PCDH17   -0.417     0.475       0.512       False  0.888
      NPY   -0.396     0.556       0.360       False  0.885
    SLIT2    0.468     0.401       0.657        True  0.883
    PAMR1   -0.246     0.840       0.425       False  0.876
    MUC13   -0.339     0.740       0.048       False  0.868
TNFRSF11A   -0.210     0.911       0.478       False  0.863
  APBB1IP    0.405     0.516       0.083        True  0.863
     CTSZ   -0.339     0.660       0.047       False  0.858

Rank of known T2D genes (out of 1024 ):

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_t2d.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.